# General Pipeline

Bu notebook, `cleaning_theory.md` kararlarina gore RAM-dostu bir data cleaning pipeline kurar.

Temel fark:
- veriyi tek seferde belleğe almiyoruz
- `pyarrow` batch'leri ile chunked cleaning yapiyoruz
- temizlenen veriyi batch batch parquet olarak disari yaziyoruz

Production calistirma onerisi:
- ayni mantigin script karsiligi olan `general_pipeline.py` kullanilsin
- script otomatik `batch_size` hesaplar ve cache'li hizli domain parser kullanir

Bu notebook cleaning katmanini kapsar. Near-duplicate, burst, coordination ve semantic risk adimlari sonraki asamalara birakilmistir.


In [9]:
from pathlib import Path
from urllib.parse import urlparse
import gc
import re
import time
import unicodedata

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

DATA_PATH = Path("datathonFINAL.parquet")
OUTPUT_PATH = Path("datathonFINAL_cleaned.parquet")

BATCH_SIZE = 100_000
MIN_TEXT_LEN = 5
RARE_LANGUAGE_THRESHOLD = 100
MISSING_PLACEHOLDERS = {"", "nan", "null", "none", "n/a"}
BLANK_SENSITIVE_COLUMNS = ["author_hash", "original_text", "english_keywords", "primary_theme"]

parquet_file = pq.ParquetFile(DATA_PATH)
print(f"Rows: {parquet_file.metadata.num_rows:,}")
print(f"Row groups: {parquet_file.metadata.num_row_groups}")
print(f"Columns: {parquet_file.schema.names}")


Rows: 5,004,813
Row groups: 528
Columns: ['original_text', 'english_keywords', 'sentiment', 'main_emotion', 'primary_theme', 'language', 'url', 'author_hash', 'date']


## Cleaning Helpers

Bu fonksiyonlar `cleaning_theory.md` icindeki kabul edilen kararlarin kod karsiligidir.


In [10]:
try:
    import tldextract
except ImportError:
    tldextract = None


def normalize_missing_scalar(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, str):
        stripped = value.strip()
        if stripped.lower() in MISSING_PLACEHOLDERS:
            return pd.NA
        return stripped
    return value


def normalize_missing_series(series):
    s = series.astype("string")
    s = s.str.strip()
    return s.mask(s.str.lower().isin(MISSING_PLACEHOLDERS), pd.NA)


def normalize_text_scalar(value):
    value = normalize_missing_scalar(value)
    if pd.isna(value):
        return pd.NA
    value = str(value).replace("\r\n", "\n").replace("\r", "\n")
    value = unicodedata.normalize("NFKC", value)
    value = re.sub(r"[^\S\n]+", " ", value)
    value = re.sub(r" *\n *", "\n", value)
    value = value.strip()
    return value if value else pd.NA


def normalize_text_series(series):
    return series.map(normalize_text_scalar)


def clean_keywords_scalar(value):
    value = normalize_missing_scalar(value)
    if pd.isna(value):
        return pd.NA
    parts = []
    seen = set()
    for item in str(value).split(","):
        token = " ".join(item.split()).strip()
        if not token:
            continue
        if token not in seen:
            seen.add(token)
            parts.append(token)
    return ", ".join(parts) if parts else pd.NA


def clean_keywords_series(series):
    return series.map(clean_keywords_scalar)


def extract_domain_parts_scalar(value):
    value = normalize_missing_scalar(value)
    if pd.isna(value):
        return pd.Series({
            "raw_url_domain": pd.NA,
            "registered_domain": pd.NA,
            "subdomain": pd.NA,
        })

    candidate = str(value).strip()
    parsed = urlparse(candidate if "://" in candidate else f"https://{candidate}")
    host = (parsed.hostname or candidate).lower().strip(".")

    if tldextract is not None:
        ext = tldextract.extract(host)
        registered_domain = ".".join(part for part in [ext.domain, ext.suffix] if part) or host
        subdomain = ext.subdomain or pd.NA
    else:
        parts = [part for part in host.split(".") if part]
        if len(parts) >= 2:
            registered_domain = ".".join(parts[-2:])
            subdomain = ".".join(parts[:-2]) or pd.NA
        else:
            registered_domain = host
            subdomain = pd.NA

    return pd.Series({
        "raw_url_domain": host,
        "registered_domain": registered_domain,
        "subdomain": subdomain,
    })


def _get_first_series(frame, column):
    value = frame[column]
    if isinstance(value, pd.DataFrame):
        return value.iloc[:, 0]
    return value


def compute_row_fingerprint(frame):
    original_text = _get_first_series(frame, "original_text").astype("string").fillna("<NA>")
    author_hash = _get_first_series(frame, "author_hash").astype("string").fillna("<NA>")
    date = _get_first_series(frame, "date").astype("string").fillna("<NA>")
    url = _get_first_series(frame, "url").astype("string").fillna("<NA>")
    row_key = (
        original_text
        + "\x1f"
        + author_hash
        + "\x1f"
        + date
        + "\x1f"
        + url
    )
    return pd.util.hash_pandas_object(row_key, index=False)


def iter_pandas_batches(path, batch_size=BATCH_SIZE):
    parquet = pq.ParquetFile(path)
    for batch in parquet.iter_batches(batch_size=batch_size):
        yield batch.to_pandas(types_mapper=pd.ArrowDtype)


## Pass 1: Language Frequencies

`rare_language` flag'i icin tum veri uzerinden dil frekanslarini hafif bir gecisle topluyoruz.


In [11]:
language_counts = pd.Series(dtype="int64")
pass1_start = time.time()

for batch_id, batch_df in enumerate(iter_pandas_batches(DATA_PATH), start=1):
    batch_lang = batch_df["language"].astype("string").str.strip().value_counts(dropna=False)
    language_counts = language_counts.add(batch_lang, fill_value=0)
    if batch_id % 10 == 0:
        print(f"Pass 1 batch {batch_id}: unique languages so far = {len(language_counts):,}")
    del batch_df, batch_lang
    gc.collect()

language_counts = language_counts.sort_values(ascending=False).astype("int64")
rare_languages = set(language_counts[language_counts < RARE_LANGUAGE_THRESHOLD].index.tolist())

print(f"Pass 1 completed in {(time.time() - pass1_start) / 60:.2f} minutes")
print(f"Rare language count: {len(rare_languages):,}")
language_counts.head(20)


Pass 1 batch 10: unique languages so far = 115
Pass 1 batch 20: unique languages so far = 117
Pass 1 batch 30: unique languages so far = 119
Pass 1 batch 40: unique languages so far = 120
Pass 1 batch 50: unique languages so far = 120
Pass 1 completed in 0.23 minutes
Rare language count: 47


language
en    3654695
es     373443
pt     235810
fr     141005
ja     116825
de      90033
ar      76740
tr      65807
it      45560
ru      37795
zh      21333
fa      15757
nl      12359
pl      11852
hi      10907
id       8689
sv       8588
uk       6086
fi       5392
hu       5080
dtype: int64

## Pass 2: Chunked Cleaning And Writing

Bu geciste cleaning uygulanir, exact technical duplicate kayitlar elenir ve temizlenmis veri batch batch yeni parquet dosyasina yazilir.


In [ ]:
seen_fingerprints = set()
writer = None
summary = {
    "raw_rows": 0,
    "written_rows": 0,
    "technical_duplicates_removed": 0,
    "missing_author_rows": 0,
    "empty_text_rows": 0,
    "short_text_rows": 0,
    "bad_date_rows": 0,
    "rare_language_rows": 0,
}

pass2_start = time.time()

if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()

for batch_id, batch_df in enumerate(iter_pandas_batches(DATA_PATH), start=1):
    batch_df.columns = [str(col) for col in batch_df.columns]
    summary["raw_rows"] += len(batch_df)

    for column in BLANK_SENSITIVE_COLUMNS:
        batch_df[column] = normalize_missing_series(batch_df[column])

    batch_df["date"] = pd.to_datetime(batch_df["date"], utc=True, errors="coerce")
    batch_df["normalized_text"] = normalize_text_series(batch_df["original_text"])
    batch_df["english_keywords_clean"] = clean_keywords_series(batch_df["english_keywords"])
    batch_df["primary_theme"] = batch_df["primary_theme"].fillna("unknown_theme")

    batch_df["author_hash_missing_flag"] = batch_df["author_hash"].isna().astype("int8")
    batch_df["is_empty_text"] = batch_df["normalized_text"].isna().astype("int8")
    batch_df["is_short_text"] = batch_df["normalized_text"].str.len().fillna(0).lt(MIN_TEXT_LEN).astype("int8")
    batch_df["is_rare_language"] = batch_df["language"].isin(rare_languages).astype("int8")

    domain_parts = batch_df["url"].map(extract_domain_parts_scalar)
    batch_df = batch_df.drop(columns=["raw_url_domain", "registered_domain", "subdomain"], errors="ignore")
    batch_df = pd.concat([batch_df, domain_parts], axis=1)

    fingerprints = compute_row_fingerprint(batch_df)
    is_duplicate = fingerprints.isin(seen_fingerprints)
    batch_df["technical_duplicate_flag"] = is_duplicate.astype("int8")

    new_fingerprints = fingerprints.loc[~is_duplicate].tolist()
    seen_fingerprints.update(new_fingerprints)

    cleaned_batch = batch_df.loc[~is_duplicate].copy()

    summary["technical_duplicates_removed"] += int(is_duplicate.sum())
    summary["missing_author_rows"] += int(cleaned_batch["author_hash_missing_flag"].sum())
    summary["empty_text_rows"] += int(cleaned_batch["is_empty_text"].sum())
    summary["short_text_rows"] += int(cleaned_batch["is_short_text"].sum())
    summary["bad_date_rows"] += int(cleaned_batch["date"].isna().sum())
    summary["rare_language_rows"] += int(cleaned_batch["is_rare_language"].sum())
    summary["written_rows"] += len(cleaned_batch)

    table = pa.Table.from_pandas(cleaned_batch, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PATH, table.schema, compression="snappy")
    writer.write_table(table)

    if batch_id % 5 == 0:
        elapsed = (time.time() - pass2_start) / 60
        print(
            f"Pass 2 batch {batch_id}: written={summary['written_rows']:,}, "
            f"duplicates_removed={summary['technical_duplicates_removed']:,}, elapsed={elapsed:.2f} min"
        )

    del batch_df, domain_parts, fingerprints, is_duplicate, cleaned_batch, table, new_fingerprints
    gc.collect()

if writer is not None:
    writer.close()

print(f"Pass 2 completed in {(time.time() - pass2_start) / 60:.2f} minutes")
print(f"Output written to: {OUTPUT_PATH}")


ValueError: Duplicate column names found: ['original_text', 'english_keywords', 'sentiment', 'main_emotion', 'primary_theme', 'language', 'url', 'author_hash', 'date', 'normalized_text', 'english_keywords_clean', 'author_hash_missing_flag', 'is_empty_text', 'is_short_text', 'is_rare_language', 'url', 'technical_duplicate_flag']

: 

## Cleaning Summary

Bu hucre, cleaning sonrasi temel kalite kontrol ozetini verir.


In [ ]:
pd.Series(summary).to_frame("value")


In [ ]:
preview_df = pd.read_parquet(OUTPUT_PATH, engine="pyarrow").head(10)
preview_df[[
    "original_text",
    "normalized_text",
    "english_keywords",
    "english_keywords_clean",
    "language",
    "is_rare_language",
    "url",
    "registered_domain",
    "subdomain",
    "author_hash",
    "author_hash_missing_flag",
    "primary_theme",
    "date",
]]


## Deferred To Later Stages

Bu cleaning notebook'u bilincli olarak su adimlari erteledi:
- near-duplicate benzerlik analizi
- burst / coordination feature'lari
- `author_hash` bos kayitlar icin ozel risk formulu
- semantic risk hesaplari

Bu konular sonraki feature engineering ve scoring notebook'larinda ele alinmali.
